# Eerste MX-experiment: alleen gewichten

**MXFP6 E2M3 / A32, blokgrootte 32.** Dit notebook leest uitsluitend opgeslagen gegevens.
Geen training, modelcalls, herkalibratie of downloads. De bron is de eerder getrainde SVHN-train-only-referentie.

De eerste proef is **directe conversie van een bestaand checkpoint**, niet QAT.
Gewichtsquantisatie is nieuw; de tanh/normalisatie van het oorspronkelijke W32-pad, A32/ReLU en BN32 blijven behouden.
Beste en laatste checkpoint blijven geselecteerd volgens de oude no-MX-run.

In [ ]:
from pathlib import Path
RUN_DIR = Path.cwd()


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
report = json.loads((RUN_DIR / 'run_report.json').read_text(encoding='utf-8'))
assert report['command'] == 'evaluate', 'Open de notebook uit de evaluate-map, niet de check-map.'
metrics = pd.read_csv(RUN_DIR / 'metrics.csv')
layers = pd.read_csv(RUN_DIR / 'layer_metrics.csv')
assert report['optimizer_steps'] == 0
assert report['config']['activations'] == 'original_A32_ReLU'
assert report['config']['weight_format'] == 'MXFP6_E2M3'
if report.get('synthetic_fixture_only', False):
    display(Markdown('## SYNTHETISCHE SOFTWARETEST - GEEN SVHN-RESULTATEN'))
display(Markdown('**Rapportstatus: ' + report['status'] + '**'))
if report['status'] != 'PASS':
    display(Markdown('**Eerst de afwijkende controles beoordelen; geen MX-conclusie trekken.**'))
display(pd.DataFrame([report['config']]).T.rename(columns={0:'Instelling'}))

## Vier paden, hetzelfde checkpoint

`reference_W32_A32`: oorspronkelijke code. `wrapper_MX_off_A32`: nieuwe adapter met MX uit.
`MXFP6_E2M3_A32`: MX aan. `stored_MXFP6_E2M3_A32`: herladen broncodes.

De twee middelste controleverschillen horen numeriek verwaarloosbaar te zijn en mogen de voorspellingen niet wijzigen.
Het verschil **MX versus reference** is de gemeten directe conversiegevoeligheid; niet het resultaat na MX-training.

In [ ]:
display(metrics[['role','epoch','mode','n','correct1','count_top1_percent','hard_cross_entropy']])
wide = metrics.pivot(index=['role','epoch'],columns='mode',values='count_top1_percent')
wide['MX_minus_reference_pp'] = wide['MXFP6_E2M3_A32'] - wide['reference_W32_A32']
display(wide)


In [ ]:
pair = wide[['reference_W32_A32','MXFP6_E2M3_A32']].copy()
pair.index = [f'{role}, epoch {epoch}' for role,epoch in pair.index]
ax = pair.plot.bar(figsize=(8,4),rot=0)
ax.set_ylabel('Testaccuracy (%)')
ax.set_ylim(0,100)
ax.set_title('Gewichts-MX zonder hertraining; activaties blijven A32')
ax.legend(['Referentie W32/A32','MXFP6 E2M3/A32'])
plt.tight_layout()
plt.show()


## Invarianties en afwijkingen

De runner rapporteert exacte logitsgelijkheid en een afzonderlijke numerieke tolerantiecontrole.
Het aantal correcte voorspellingen van de no-MX-hermeting moet overeenkomen met het broncheckpoint.
Een andere score is eerst een reproduceerbaarheidsvraag, geen effect van MX.

In [ ]:
display(pd.DataFrame(report['checks']).T)
display(Markdown('### Waarschuwingen'))
display(report.get('warnings', []))


## Lokale quantisatiefout en code-opslag

MAE/RMSE vergelijken de gedecodeerde MX-gewichten met de EFFECTIEVE W32-gewichten, niet met de onbewerkte master.
`new_zero_count` telt niet-nulbronnen die nul worden. `local_subnormal_count` betreft E2M3-subnormals.
Schaalclipping en subnormale FP32-bronnen worden apart gerapporteerd.

In [ ]:
display(layers)
storage = layers.groupby(['role','epoch'])[['n_weights','padded_slots','packed_weight_and_scale_bytes','fp32_weight_bytes']].sum()
storage['auxiliary_state_bytes'] = [report['checks'][role]['auxiliary_state_bytes'] for role,epoch in storage.index]
storage['payload_bytes_including_auxiliary'] = storage['packed_weight_and_scale_bytes'] + storage['auxiliary_state_bytes']
display(storage)


**Geen hardwareclaim:** de MX-export bevat 24 codebytes + 1 schaalbyte per 32 gewichten, plus padding en aanvullende staat.
De bestandsheader en de FP32-rekenbuffers van de emulator zijn andere grootheden.
Deze meting bewijst geen native 6-bitversnelling, energiewinst, trainingsconvergentie of truncatierobuustheid.

Bronnen: auteurscode in de bijgeleverde bron-ZIP; projectpaper *Microscaling Data Formats for Deep Learning*, p. 2-4, tabel 1/Algorithm 1;
OCP MX-specificatie v1.0 (2023), p. 11-14. De experimentele keuzes staan in `CODEWIJZIGINGEN.md`.